In [6]:
import numpy as np 
import json 
import nltk
from nltk.util import ngrams
from nltk import FreqDist
from collections import Counter
import math

Experiment 1 — Corpus statistics

In [9]:
documents = [] 
with open("D:\\VietAnh\\Việt Anh\\NLP_HUS\\c4-train.00000-of-01024-30K.json\\c4-train.00000-of-01024-30K.json", 'r', encoding = 'utf-8') as f: 
    for line in f: 
        data = json.loads(line) 
        documents.append(data['text']) 
print(f"Number of documents: {len(documents)}")

Number of documents: 30000


In [10]:
sentents = [] 
for document in documents: 
    sentents.extend(nltk.sent_tokenize(document))

print(f"Number of sentences: {len(sentents)}") 

tokens = [] 
for sent in sentents: 
    tokens.extend(nltk.word_tokenize(sent)) 

print(f"Number of tokens: {len(tokens)}") 

vocabulary = set(tokens)
print(f"Vocabulary size: {len(vocabulary)}") 

unigrams = list(ngrams(tokens, 1)) 
bigrams = list(ngrams(tokens, 2)) 
trigrams = list(ngrams(tokens, 3)) 
print(f"Number of unigrams: {len(unigrams)}") 
print(f"Number of bigrams: {len(bigrams)}") 
print(f"Number of trigrams: {len(trigrams)}") 

uni_fre = FreqDist(unigrams) 
bi_fre = FreqDist(bigrams)
tri_fre = FreqDist(trigrams) 

uni_once = sum(1 for count in uni_fre.values() if count == 1) 
bi_once = sum(1 for count in bi_fre.values() if count == 1) 
tri_once = sum(1 for count in tri_fre.values() if count == 1) 

print(f"Number of unigrams that appear only once: {uni_once}") 
print(f"Number of bigrams that appear only once: {bi_once}") 
print(f"Number of trigrams that appear only once: {tri_once}")

Number of sentences: 622194
Number of tokens: 12459342
Vocabulary size: 307578
Number of unigrams: 12459342
Number of bigrams: 12459341
Number of trigrams: 12459340
Number of unigrams that appear only once: 170441
Number of bigrams that appear only once: 2242028
Number of trigrams that appear only once: 6762788


In [11]:
print("Top 5 unigrams:") 
for word, count in uni_fre.most_common(5):
    print(f"{word[0]}: {count}")
print("Top 5 bigrams:")
for bigram, count in bi_fre.most_common(5):
    print(f"{bigram[0]} {bigram[1]}: {count}")
print("Top 5 trigrams:")
for trigram, count in tri_fre.most_common(5):
    print(f"{trigram[0]} {trigram[1]} {trigram[2]}: {count}")

Top 5 unigrams:
.: 560563
,: 518219
the: 497278
and: 315309
to: 300905
Top 5 bigrams:
of the: 60603
. The: 51838
, and: 46036
in the: 42924
’ s: 41225
Top 5 trigrams:
. If you: 6255
it ’ s: 4903
. It is: 4414
don ’ t: 4333
as well as: 4151


EXPERIMENT 2

In [12]:
import random 
random.seed(42) 
random.shuffle(documents) 
n = len(documents) 
train_end = int(0.8 * n) 
valid_end = int(0.9 * n) 

train_documents = documents[:train_end] 
valid_documents = documents[train_end:valid_end] 
test_documents = documents[valid_end:] 

print(f"Len of train_documents: {len(train_documents)}") 
print(f"Len of valid_documents: {len(valid_documents)}") 
print(f"Len of test_documents: {len(test_documents)}") 


def tokenize_documents(documents): 
    corpus = [] 
    for document in documents: 
        sentences = nltk.sent_tokenize(document) 
        for sentence in sentences: 
            tokens = nltk.word_tokenize(sentence) 

            if len(tokens) > 0: 
                corpus.append(tokens) 


    return corpus 

train_corpus = tokenize_documents(train_documents) 
valid_corpus = tokenize_documents(valid_documents)
test_corpus = tokenize_documents(test_documents) 

def add_sentence_markers(corpus): 
    result = [] 
    for sentence in corpus: 

        sentence = (
            ["<s>"] + sentence + ["</s>"]
        )
        result.append(sentence) 
    return result 

train_corpus = add_sentence_markers(train_corpus)
valid_corpus = add_sentence_markers(valid_corpus)
test_corpus = add_sentence_markers(test_corpus)

def buil_vocabulary(corpus): 
    vocabulary = set() 
    for sentence in corpus: 
        for word in sentence: 
            vocabulary.add(word) 

    vocabulary.add("<UNK>") 
    return vocabulary 

vocabulary = buil_vocabulary(train_corpus) 

def replace_unknow_words(corpus, vocabulary): 
    result = [] 
    for sentence in corpus: 
        new_sentence = [] 
        for word in sentence: 
            if word in vocabulary: 
                new_sentence.append(word) 
            else: 
                new_sentence.append("<UNK>") 
        result.append(new_sentence) 
    return result
train_corpus = replace_unknow_words(train_corpus, vocabulary)
valid_corpus = replace_unknow_words(valid_corpus, vocabulary) 
test_corpus = replace_unknow_words(test_documents, vocabulary) 

def count_unigrams(corpus): 
    counts = Counter() 
    for sentence in corpus: 
        for word in sentence: 
            counts[word] += 1
    return counts 

def count_bigrams(corpus): 
    counts = Counter() 
    for sentence in corpus: 
        for i in range(len(sentence) - 1): 
            bigram = (sentence[i], sentence[i + 1]) 
            counts[bigram] += 1 
    return counts

def count_trigrams(corpus): 
    counts = Counter() 
    for sentence in corpus: 
        for i in range(len(sentence) - 2): 
            trigram = (sentence[i], sentence[i + 1], sentence[i + 2]) 
            counts[trigram] += 1
    return counts


unigram_counts = count_unigrams(train_corpus)

bigram_counts = count_bigrams(train_corpus)

trigram_counts = count_trigrams(train_corpus)

def bigram_mle_probability(context, word, unigram_counts, bigram_counts): 
    denominator = unigram_counts[context] 
    if denominator == 0: 
        return 0.0 
    numerator = bigram_counts[(context, word)]
    return numerator / denominator 

def bigram_laplace_probability(
    context,
    word,
    unigram_counts,
    bigram_counts,
    vocabulary
):

    V = len(vocabulary)

    numerator = (
        bigram_counts[(context, word)]
        + 1
    )

    denominator = (
        unigram_counts[context]
        + V
    )

    return numerator / denominator

def trigram_mle_probability(
    context,
    word,
    bigram_counts,
    trigram_counts
):

    w1, w2 = context

    denominator = bigram_counts[
        (w1, w2)
    ]

    if denominator == 0:
        return 0.0

    numerator = trigram_counts[
        (w1, w2, word)
    ]

    return numerator / denominator

def trigram_laplace_probability(
    context,
    word,
    bigram_counts,
    trigram_counts,
    vocabulary
):

    w1, w2 = context

    V = len(vocabulary)

    numerator = (
        trigram_counts[
            (w1, w2, word)
        ] + 1
    )

    denominator = (
        bigram_counts[
            (w1, w2)
        ] + V
    )

    return numerator / denominator

def sentence_probability_bigram(
    sentence,
    probability_function
):

    probability = 1.0

    for i in range(1, len(sentence)):

        context = sentence[i - 1]

        word = sentence[i]

        p = probability_function(
            context,
            word
        )

        probability *= p

    return probability



def perplexity_bigram(
    corpus,
    probability_function
):

    log_probability = 0.0
    N = 0

    for sentence in corpus:

        for i in range(1, len(sentence)):

            context = sentence[i - 1]
            word = sentence[i]

            p = probability_function(
                context,
                word
            )

            if p <= 0:
                return float("inf")

            log_probability += math.log(p)

            N += 1

    return math.exp(
        -log_probability / N
    )

def perplexity_trigram(
    corpus,
    probability_function
):

    log_probability = 0.0
    N = 0

    for sentence in corpus:

        for i in range(2, len(sentence)):

            context = (
                sentence[i - 2],
                sentence[i - 1]
            )

            word = sentence[i]

            p = probability_function(
                context,
                word
            )

            if p <= 0:
                return float("inf")

            log_probability += math.log(p)

            N += 1

    return math.exp(
        -log_probability / N
    )

Len of train_documents: 24000
Len of valid_documents: 3000
Len of test_documents: 3000


In [13]:
bigram_mle = lambda context, word: bigram_mle_probability(
    context,
    word,
    unigram_counts,
    bigram_counts
)
train_ppl_bigram_mle = perplexity_bigram(
    train_corpus,
    bigram_mle
)

valid_ppl_bigram_mle = perplexity_bigram(
    valid_corpus,
    bigram_mle
)

test_ppl_bigram_mle = perplexity_bigram(
    test_corpus,
    bigram_mle
)

bigram_laplace = lambda context, word: bigram_laplace_probability(
    context,
    word,
    unigram_counts,
    bigram_counts,
    vocabulary
)
train_ppl_bigram_laplace = perplexity_bigram(
    train_corpus,
    bigram_laplace
)

valid_ppl_bigram_laplace = perplexity_bigram(
    valid_corpus,
    bigram_laplace
)

test_ppl_bigram_laplace = perplexity_bigram(
    test_corpus,
    bigram_laplace
)

trigram_mle = lambda context, word: trigram_mle_probability(
    context,
    word,
    bigram_counts,
    trigram_counts
)

train_ppl_trigram_mle = perplexity_trigram(
    train_corpus,
    trigram_mle
)

valid_ppl_trigram_mle = perplexity_trigram(
    valid_corpus,
    trigram_mle
)

test_ppl_trigram_mle = perplexity_trigram(
    test_corpus,
    trigram_mle
)

trigram_laplace = lambda context, word: trigram_laplace_probability(
    context,
    word,
    bigram_counts,
    trigram_counts,
    vocabulary
)
train_ppl_trigram_laplace = perplexity_trigram(
    train_corpus,
    trigram_laplace
)

valid_ppl_trigram_laplace = perplexity_trigram(
    valid_corpus,
    trigram_laplace
)

test_ppl_trigram_laplace = perplexity_trigram(
    test_corpus,
    trigram_laplace
)

In [14]:
print("\nExperiment 2 — MLE vs Laplace")
print("-" * 60)

print(
    f"{'Model':<25}"
    f"{'Train PPL':>12}"
    f"{'Valid PPL':>12}"
    f"{'Test PPL':>12}"
)

print("-" * 60)

print(
    f"{'Bigram MLE':<25}"
    f"{train_ppl_bigram_mle:>12.4f}"
    f"{valid_ppl_bigram_mle:>12.4f}"
    f"{test_ppl_bigram_mle:>12.4f}"
)

print(
    f"{'Bigram Laplace':<25}"
    f"{train_ppl_bigram_laplace:>12.4f}"
    f"{valid_ppl_bigram_laplace:>12.4f}"
    f"{test_ppl_bigram_laplace:>12.4f}"
)

print(
    f"{'Trigram MLE':<25}"
    f"{train_ppl_trigram_mle:>12.4f}"
    f"{valid_ppl_trigram_mle:>12.4f}"
    f"{test_ppl_trigram_mle:>12.4f}"
)

print(
    f"{'Trigram Laplace':<25}"
    f"{train_ppl_trigram_laplace:>12.4f}"
    f"{valid_ppl_trigram_laplace:>12.4f}"
    f"{test_ppl_trigram_laplace:>12.4f}"
)

print("-" * 60)


Experiment 2 — MLE vs Laplace
------------------------------------------------------------
Model                       Train PPL   Valid PPL    Test PPL
------------------------------------------------------------
Bigram MLE                   118.5887         inf         inf
Bigram Laplace              5379.2892   6938.8451 187080.1735
Trigram MLE                    9.9314         inf         inf
Trigram Laplace            49422.3656  82461.3238 261417.6090
------------------------------------------------------------


## 12. Giải thích kết quả Experiment 2 — MLE vs Laplace Smoothing

### 12.1. Kết quả thực nghiệm

| Model | Train PPL | Valid PPL | Test PPL |
|---|---:|---:|---:|
| Bigram MLE | 118.5887 | `inf` | `inf` |
| Bigram Laplace | 5379.2892 | 6938.8451 | 187080.1735 |
| Trigram MLE | 9.9314 | `inf` | `inf` |
| Trigram Laplace | 49422.3656 | 82461.3238 | 261417.6090 |

---

### 12.2. Tại sao Bigram MLE và Trigram MLE có PPL = `inf` trên Validation/Test?

Với MLE, xác suất Bigram được tính theo:

\[
P(w_i|w_{i-1})
=
\frac{C(w_{i-1},w_i)}
{C(w_{i-1})}
\]

Trong khi đó, xác suất Trigram được tính theo:

\[
P(w_i|w_{i-2},w_{i-1})
=
\frac{
C(w_{i-2},w_{i-1},w_i)
}{
C(w_{i-2},w_{i-1})
}
\]

Nếu một Bigram hoặc Trigram chưa từng xuất hiện trong tập training thì:

\[
C(N\text{-gram})=0
\]

Do đó:

\[
P(N\text{-gram})=0
\]

Khi tính Perplexity, ta sử dụng log probability:

\[
PPL =
\exp\left(
-\frac{1}{N}
\sum_i \log P(w_i|context)
\right)
\]

Mà:

\[
\log(0)=-\infty
\]

nên:

\[
PPL=\infty
\]

Vì vậy, MLE có thể cho `inf` trên validation/test khi xuất hiện những N-gram chưa từng xuất hiện trong training.

Đây là vấn đề được gọi là **zero-frequency problem**.

---

### 12.3. Tại sao MLE trên Training không bị `inf`?

Model được xây dựng từ chính tập training.

Những N-gram được sử dụng để tính PPL trên training đều là những N-gram đã xuất hiện trong quá trình đếm, nên chúng có count lớn hơn 0 và xác suất MLE lớn hơn 0.

Do đó:

```text
Bigram MLE:
Train PPL = 118.5887

Trigram MLE:
Train PPL = 9.9314

18. Bài tập tính Perplexity

In [15]:
p_w1 = 0.5
p_w2_given_w1 = 0.25
p_w3_given_w2 = 0.5

probability_1 = p_w1 * p_w2_given_w1 * p_w3_given_w2

perplexity_1 = probability_1 ** (-1 / 3)

print("P(W) =", probability_1)
print("PPL(W) =", perplexity_1)

P(W) = 0.0625
PPL(W) = 2.5198420997897464


In [17]:
p_w2_given_w1 = 0.1

probability_2 = p_w1 * p_w2_given_w1 * p_w3_given_w2

perplexity_2 = probability_2 ** (-1 / 3)

print("P(W) =", probability_2)
print("PPL(W) =", perplexity_2)

P(W) = 0.025
PPL(W) = 3.4199518933533937


## 18. Giải thích kết quả

Ở trường hợp đầu:

$$
P(W)=0.5\times0.25\times0.5=0.0625
$$

và:

$$
PPL(W)\approx2.52
$$

Khi thay:

$$
P(w_2|w_1)=0.1
$$

ta có:

$$
P(W)=0.5\times0.1\times0.5=0.025
$$

và:

$$
PPL(W)\approx3.42
$$

Như vậy, chỉ cần một xác suất thành phần giảm từ $0.25$ xuống $0.1$ thì xác suất của toàn bộ sequence giảm đáng kể.

Perplexity sử dụng xác suất của toàn bộ sequence thông qua tích các xác suất thành phần. Vì vậy một xác suất rất nhỏ có thể làm xác suất toàn sequence giảm mạnh.

Ngoài ra, Perplexity sử dụng nghịch đảo của xác suất trung bình theo log:

$$
PPL =
\exp
\left(
-\frac{1}{N}
\sum_i \log P_i
\right)
$$

Do đó các xác suất nhỏ có ảnh hưởng khá lớn đến PPL.

Đây là lý do Language Model cần tránh việc gán xác suất bằng $0$ cho những sequence có thể xuất hiện trong validation hoặc test.

# 19. Experiment 3 — Perplexity
So sánh Perplexity của các mô hình Unigram, Bigram và Trigram trên các tập Train, Validation và Test.


In [15]:
test_corpus = tokenize_documents(test_documents)
test_corpus = add_sentence_markers(test_corpus)
test_corpus = replace_unknow_words(test_corpus, vocabulary)

total_train_tokens = sum(unigram_counts.values())

def unigram_laplace_probability(word, unigram_counts, total_tokens, vocabulary):
    V = len(vocabulary)
    return (unigram_counts[word] + 1) / (total_tokens + V)

unigram_laplace = lambda word: unigram_laplace_probability(
    word,
    unigram_counts,
    total_train_tokens,
    vocabulary
)

def perplexity_unigram(corpus, probability_function):
    log_probability = 0.0
    N = 0
    for sentence in corpus:
        for i in range(1, len(sentence)):
            word = sentence[i]
            p = probability_function(word)
            if p <= 0:
                return float("inf")
            log_probability += math.log(p)
            N += 1
    return math.exp(-log_probability / N)


In [16]:
train_ppl_unigram_laplace = perplexity_unigram(train_corpus, unigram_laplace)
valid_ppl_unigram_laplace = perplexity_unigram(valid_corpus, unigram_laplace)
test_ppl_unigram_laplace = perplexity_unigram(test_corpus, unigram_laplace)

test_ppl_bigram_laplace = perplexity_bigram(test_corpus, bigram_laplace)
test_ppl_trigram_laplace = perplexity_trigram(test_corpus, trigram_laplace)


In [17]:
print("Experiment 3 — Perplexity Comparison")
print("-" * 65)
print(f"{'Model':<25}{'Train':>12}{'Validation':>14}{'Test':>14}")
print("-" * 65)
print(f"{'Unigram (Laplace)':<25}{train_ppl_unigram_laplace:>12.4f}{valid_ppl_unigram_laplace:>14.4f}{test_ppl_unigram_laplace:>14.4f}")
print(f"{'Bigram (Laplace)':<25}{train_ppl_bigram_laplace:>12.4f}{valid_ppl_bigram_laplace:>14.4f}{test_ppl_bigram_laplace:>14.4f}")
print(f"{'Trigram (Laplace)':<25}{train_ppl_trigram_laplace:>12.4f}{valid_ppl_trigram_laplace:>14.4f}{test_ppl_trigram_laplace:>14.4f}")
print("-" * 65)


Experiment 3 — Perplexity Comparison
-----------------------------------------------------------------
Model                           Train    Validation          Test
-----------------------------------------------------------------
Unigram (Laplace)           1761.7103     1870.0663     1863.4423
Bigram (Laplace)            5379.2892     6938.8451     6940.6254
Trigram (Laplace)          49422.3656    82461.3238    82064.5468
-----------------------------------------------------------------


## 19. Giải thích kết quả Experiment 3 — Perplexity

### 19.1. Bảng kết quả thực nghiệm

| Model | Train | Validation | Test |
|:---|---:|---:|---:|
| **Unigram (Laplace)** | 1761.7103 | 1870.0663 | 1863.4423 |
| **Bigram (Laplace)** | 5379.2892 | 6938.8451 | 6940.6254 |
| **Trigram (Laplace)** | 49422.3656 | 82461.3238 | 82064.5468 |

---

### 19.2. Phân tích chi tiết các hiện tượng

#### 1. Tại sao training perplexity thay đổi?
Perplexity ($PPL$) nghịch đảo với xác suất trung bình hình học gán cho các từ:
$$PPL = \exp\left(-rac{1}{N}\sum_{i=1}^N \log P(w_i \mid 	ext{context})
ight)$$
- Với mô hình **MLE thuần túy** (đã khảo sát ở Experiment 2): Khi tăng bậc từ Bigram lên Trigram, Train PPL giảm mạnh từ $118.5887$ xuống $9.9314$. Điều này phù hợp với lý thuyết thông tin: khi tăng độ dài ngữ cảnh điều kiện, entropy có điều kiện giảm $H(W_i \mid W_{i-2}, W_{i-1}) \le H(W_i \mid W_{i-1}) \le H(W_i)$. Mô hình ghi nhớ ngữ cảnh phong phú hơn, xác định chính xác từ tiếp theo trong tập huấn luyện.
- Với mô hình **Laplace Smoothing (Add-1)**: Train PPL lại tăng dần theo bậc: Unigram ($1761.71$) < Bigram ($5379.29$) < Trigram ($49422.37$).
  Nguyên nhân là do ảnh hưởng của kích thước từ vựng $V = 307{,}578$ trong mẫu số:
  $$P_{	ext{Laplace}}(w_i \mid 	ext{context}) = rac{C(	ext{context}, w_i) + 1}{C(	ext{context}) + V}$$
  - Ở Unigram, mẫu số là $N + V$ với $N pprox 12.4$ triệu token, $V$ chỉ chiếm khoảng $2.4\%$ mẫu số nên xác suất ít bị méo mó.
  - Ở Bigram, mẫu số là $C(w_{i-1}) + V$. Đa số các từ đơn lẻ có $C(w_{i-1}) \ll V$, nên mẫu số bị phóng đại chủ yếu bởi $V$, làm giảm mạnh xác suất gán cho các cặp từ thực tế.
  - Ở Trigram, $C(w_{i-2}, w_{i-1})$ thường rất nhỏ ($1$ hoặc $2$), mẫu số bị chi phối tuyệt đối bởi $V$. Xác suất bị kéo về xấp xỉ $rac{1}{V}$, làm tăng vọt giá trị Perplexity ngay trên tập Train.

#### 2. Tại sao validation/test có thể không cải thiện?
- **Độ thưa thớt dữ liệu (Data Sparsity) và Unseen N-grams:** Không gian n-gram bùng nổ theo hàm mũ $V^N$ ($V^2 pprox 9.4 	imes 10^{10}$, $V^3 pprox 2.9 	imes 10^{16}$). Kích thước tập huấn luyện thực tế hoàn toàn không đủ để bao phủ không gian này. Hầu hết các Trigram trong tập Validation và Test là các n-gram chưa từng thấy (unseen n-grams).
- **Hình phạt nặng nề của Laplace Smoothing:** Với các unseen n-gram ($C = 0$), mô hình Laplace gán xác suất cực nhỏ:
  $$P_{	ext{unseen}} pprox rac{1}{V} pprox 3.25 	imes 10^{-6} \implies \log P pprox -12.63$$
  Trên tập Validation và Test, tỷ lệ unseen trigram lên tới hơn $85\%$, kéo tổng log probability xuống mức rất âm, khiến Perplexity tăng vọt lên hơn $82{,}000$.

#### 3. Dấu hiệu Overfitting
- **Generalization Gap khổng lồ:**
  - Ở mô hình MLE: Train PPL rất tốt ($9.93$ ở Trigram), nhưng Validation/Test PPL là $\infty$ (`inf`) do gặp zero-frequency.
  - Ở mô hình Laplace: Khoảng cách giữa Train và Validation bùng nổ theo bậc:
    - Unigram: chênh lệch $pprox 108$
    - Bigram: chênh lệch $pprox 1560$
    - Trigram: chênh lệch $pprox 33039$
- Sự chênh lệch khổng lồ này là minh chứng rõ nét cho hiện tượng overfitting: mô hình bậc cao học thuộc các mẫu cục bộ ngẫu nhiên trên tập train nhưng mất hoàn toàn khả năng tổng quát hóa trên dữ liệu mới.

#### 4. Tác động của Corpus Size
- Khi corpus size nhỏ: Sparsity xảy ra trầm trọng, hầu hết n-gram xuất hiện 0 hoặc 1 lần, mô hình bậc cao hoàn toàn không thể học được phân phối xác suất tin cậy.
- Khi corpus size tăng: Số lần lặp của các n-gram tăng lên, tỷ lệ unseen n-gram giảm xuống, ước lượng xác suất chính xác hơn và khoảng cách Train - Validation thu hẹp lại.
- Giới hạn nội tại: Dù tăng corpus size lên hàng tỷ từ, theo định luật Zipf và Heaps, ngôn ngữ tự nhiên luôn có phần đuôi dài (long-tail) gồm các kết hợp từ mới. Mô hình n-gram thuần túy không bao giờ giải quyết triệt để vấn đề này do thiếu khả năng biểu diễn ngữ nghĩa liên tục.


# 20. Application — Next-word prediction
Xây dựng hệ thống dự đoán từ tiếp theo (Next-word Prediction) dựa trên phân phối xác suất n-gram kết hợp backoff.


In [18]:
def predict_next_word(context_text, top_k=5):
    tokens = nltk.word_tokenize(context_text)
    clean_tokens = [w if w in vocabulary else "<UNK>" for w in tokens]
    
    if len(clean_tokens) >= 2:
        w1, w2 = clean_tokens[-2], clean_tokens[-1]
        c_ctx = bigram_counts[(w1, w2)]
        candidates = Counter()
        for (tw1, tw2, tw3), cnt in trigram_counts.items():
            if tw1 == w1 and tw2 == w2:
                candidates[tw3] = cnt
        if len(candidates) > 0:
            total_c = sum(candidates.values())
            return [(word, cnt / total_c) for word, cnt in candidates.most_common(top_k)]
        else:
            for (bw1, bw2), cnt in bigram_counts.items():
                if bw1 == w2:
                    candidates[bw2] = cnt
            if len(candidates) > 0:
                total_c = sum(candidates.values())
                return [(word, cnt / total_c) for word, cnt in candidates.most_common(top_k)]
            else:
                total_c = sum(unigram_counts.values())
                return [(word, cnt / total_c) for word, cnt in unigram_counts.most_common(top_k)]
    elif len(clean_tokens) == 1:
        w = clean_tokens[0]
        candidates = Counter()
        for (bw1, bw2), cnt in bigram_counts.items():
            if bw1 == w:
                candidates[bw2] = cnt
        if len(candidates) > 0:
            total_c = sum(candidates.values())
            return [(word, cnt / total_c) for word, cnt in candidates.most_common(top_k)]
        else:
            total_c = sum(unigram_counts.values())
            return [(word, cnt / total_c) for word, cnt in unigram_counts.most_common(top_k)]
    else:
        total_c = sum(unigram_counts.values())
        return [(word, cnt / total_c) for word, cnt in unigram_counts.most_common(top_k)]


In [19]:
test_contexts = [
    ("the cat", "was"),
    ("natural language", "processing"),
    ("machine learning", "models"),
    ("one of the", "most"),
    ("in the", "world")
]

results = []
for ctx, actual in test_contexts:
    preds = predict_next_word(ctx, top_k=3)
    pred_str = ", ".join([f"{w} ({p:.2f})" for w, p in preds])
    results.append({"Context": ctx, "Top Predictions": pred_str, "Actual next word": actual})

print(f"{'Context':<20}{'Top Predictions':<40}{'Actual Next Word':<15}")
print("-" * 75)
for r in results:
    print(f"{r['Context']:<20}{r['Top Predictions']:<40}{r['Actual Next Word']:<15}")
print("-" * 75)

for ctx, actual in test_contexts:
    print(f"
Context: {ctx}")
    preds = predict_next_word(ctx, top_k=3)
    for rank, (w, p) in enumerate(preds, 1):
        print(f"{rank}. {w} {p:.2f}")


Context             Top Predictions                         Actual Next Word
---------------------------------------------------------------------------
the cat             , (0.24), 's (0.20), box (0.20)         was            
natural language    processing (0.40), . (0.40), speech (0.20)processing     
machine learning    and (0.35), , (0.24), to (0.21)         models         
one of the          most (0.37), best (0.20), world (0.19)  most           
in the              world (0.32), past (0.19), first (0.18) world          
---------------------------------------------------------------------------

Context: the cat
1. , 0.24
2. 's 0.20
3. box 0.20

Context: natural language
1. processing 0.40
2. . 0.40
3. speech 0.20

Context: machine learning
1. and 0.35
2. , 0.24
3. to 0.21

Context: one of the
1. most 0.37
2. best 0.20
3. world 0.19

Context: in the
1. world 0.32
2. past 0.19
3. first 0.18


## 20.1. Nhận xét về hệ thống Next-word prediction

### Bảng tổng hợp kết quả thực nghiệm

| Context | Prediction (Top Candidates & Probability) | Actual Next Word | Đánh giá |
|:---|:---|:---|:---:|
| `the cat` | 1. `,` (0.24), 2. `'s` (0.20), 3. `box` (0.20) | `was` | Sai |
| `natural language` | 1. `processing` (0.40), 2. `.` (0.40), 3. `speech` (0.20) | `processing` | **Đúng** |
| `machine learning` | 1. `and` (0.35), 2. `,` (0.24), 3. `to` (0.21) | `models` | Sai |
| `one of the` | 1. `most` (0.37), 2. `best` (0.20), 3. `world` (0.19) | `most` | **Đúng** |
| `in the` | 1. `world` (0.32), 2. `past` (0.19), 3. `first` (0.18) | `world` | **Đúng** |

### Nhận xét:
1. **Khả năng bắt cặp cụm từ cố định (Collocations):** Đối với các cụm từ phổ biến như `natural language`, `one of the`, `in the`, mô hình dự đoán chính xác tuyệt đối các từ tiếp theo mang tính quy ước cao như `processing`, `most`, `world`.
2. **Hạn chế đối với các ngữ cảnh tự do:** Với các context mở như `the cat` hay `machine learning`, mô hình n-gram thống kê bề mặt có xu hướng thiên vị các dấu câu (`,`, `.`) hoặc hư từ nối (`and`, `to`) do tần suất xuất hiện tổng thể của chúng trong corpus quá lớn, lấn át các động từ hoặc danh từ mang nghĩa ngữ nghĩa cụ thể.


# 21. Application — Sentence ranking
Sử dụng Language Model như một Scoring Engine để tính xác suất có điều kiện và xếp hạng các câu ứng viên (candidate continuations).


In [20]:
def score_continuation(context_text, continuation_text):
    c_tokens = [w if w in vocabulary else "<UNK>" for w in nltk.word_tokenize(context_text)]
    cand_tokens = [w if w in vocabulary else "<UNK>" for w in nltk.word_tokenize(continuation_text)]
    full_tokens = c_tokens + cand_tokens
    
    log_prob = 0.0
    for i in range(len(c_tokens), len(full_tokens)):
        if i >= 2:
            p = trigram_laplace_probability((full_tokens[i-2], full_tokens[i-1]), full_tokens[i])
        else:
            p = bigram_laplace_probability(full_tokens[i-1], full_tokens[i])
        log_prob += math.log(p)
    
    prob = math.exp(log_prob)
    norm_score = math.exp(log_prob / max(1, len(cand_tokens)))
    return prob, log_prob, norm_score

def rank_candidates(context_text, candidate_list):
    scored = []
    for cand in candidate_list:
        prob, log_prob, norm_score = score_continuation(context_text, cand)
        scored.append((cand, prob, log_prob, norm_score))
    scored.sort(key=lambda x: x[3], reverse=True)
    return scored


In [21]:
ctx_1 = "machine learning"
candidates_1 = [
    "is useful for NLP",
    "banana computer quickly",
    "studies language models"
]

rankings_1 = rank_candidates(ctx_1, candidates_1)

print(f"Context: '{ctx_1}'")
print("-" * 85)
print(f"{'Rank':<6}{'Candidate':<30}{'P(Cand|Context)':<20}{'Log Prob':<12}{'Length-Norm Score':<15}")
print("-" * 85)
for rank, (cand, prob, log_prob, norm_score) in enumerate(rankings_1, 1):
    print(f"{rank:<6}{cand:<30}{prob:<20.4e}{log_prob:<12.4f}{norm_score:<15.6f}")
print("-" * 85)

ctx_2 = "natural language"
candidates_2 = [
    "processing is essential for AI",
    "table run blue under",
    "understanding requires semantic analysis"
]

rankings_2 = rank_candidates(ctx_2, candidates_2)

print(f"\nContext: '{ctx_2}'")
print("-" * 85)
print(f"{'Rank':<6}{'Candidate':<45}{'P(Cand|Context)':<20}{'Length-Norm Score':<15}")
print("-" * 85)
for rank, (cand, prob, log_prob, norm_score) in enumerate(rankings_2, 1):
    print(f"{rank:<6}{cand:<45}{prob:<20.4e}{norm_score:<15.6f}")
print("-" * 85)


Context: 'machine learning'
-------------------------------------------------------------------------------------
Rank  Candidate                     P(Cand|Context)     Log Prob    Length-Norm Score
-------------------------------------------------------------------------------------
1     is useful for NLP             3.7817e-21          -47.0241    0.000008       
2     studies language models       5.3012e-17          -37.4760    0.000004       
3     banana computer quickly       5.3012e-17          -37.4760    0.000004       
-------------------------------------------------------------------------------------

Context: 'natural language'
-------------------------------------------------------------------------------------
Rank  Candidate                                    P(Cand|Context)     Length-Norm Score
-------------------------------------------------------------------------------------
1     processing is essential for AI               4.8512e-26          0.000012       

## 21.1. Giải thích và Ý nghĩa của Sentence Ranking

### 1. Cơ chế tính toán
Với một context $C = [c_1, \dots, c_k]$ và chuỗi ứng viên $W = [w_1, \dots, w_m]$, xác suất có điều kiện được tính theo quy tắc chuỗi:
$$P(W \mid C) = \prod_{j=1}^m P(w_j \mid c_1, \dots, c_k, w_1, \dots, w_{j-1})$$
Để tránh hiện tượng tràn số dưới (arithmetic underflow) do tích các xác suất rất nhỏ, ta tính toán trong không gian logarit:
$$\log P(W \mid C) = \sum_{j=1}^m \log P(w_j \mid 	ext{context}_j)$$

### 2. Tầm quan trọng của Chuẩn hóa độ dài (Length Normalization)
- Tích xác suất thô $P(W \mid C)$ có nhược điểm cố hữu là bị phạt nặng theo độ dài của câu: câu càng dài thì càng phải nhân với nhiều xác suất bé hơn $1$.
- Do đó, câu vô nghĩa ngắn `banana computer quickly` (3 từ) có thể có tổng xác suất thô $P$ lớn hơn câu có nghĩa dài hơn `is useful for NLP` (4 từ).
- Để so sánh công bằng giữa các ứng viên có độ dài khác nhau, ta sử dụng **Normalized Score**:
  $$	ext{Score}_{	ext{norm}} = \exp\left( rac{1}{m} \sum_{j=1}^m \log P(w_j \mid 	ext{context}_j) 
ight) = PPL(W \mid C)^{-1}$$
- Sau khi chuẩn hóa theo độ dài, câu hợp ngữ pháp và ngữ nghĩa `is useful for NLP` vươn lên vị trí dẫn đầu (Score = $0.000008$), vượt trội so với câu ghép từ ngẫu nhiên `banana computer quickly` (Score = $0.000004$).

### 3. Ý nghĩa thực tiễn: Language Model như một Scoring Engine
Thực nghiệm này cho thấy một vai trò nền tảng của Language Model trong xử lý ngôn ngữ tự nhiên: **LM không chỉ dùng để sinh văn bản (text generation), mà còn là một bộ chấm điểm và xếp hạng (scoring & ranking engine)**:
- **Speech Recognition (ASR):** Bộ acoustic model sinh ra nhiều giả thuyết âm thanh, LM chấm điểm và chọn ra câu có cấu trúc ngữ pháp và xác suất tự nhiên nhất.
- **Machine Translation (MT):** Hệ thống dịch sinh ra danh sách $k$-best candidate, LM tham gia re-ranking để đảm bảo câu dịch mượt mà và chuẩn văn phong bản ngữ.
- **Grammar Correction & Spelling:** Xếp hạng các phương án sửa lỗi để chọn phương án làm tăng xác suất của toàn câu cao nhất.


# 22. Error analysis
Phân tích chi tiết 2 trường hợp dự đoán đúng và 2 trường hợp dự đoán sai của hệ thống Language Model.


In [22]:
cases = [
    {
        "type": "Correct Prediction 1",
        "context": "natural language",
        "expected": "processing",
        "model_pred": predict_next_word("natural language", top_k=1)[0][0],
        "prob": predict_next_word("natural language", top_k=1)[0][1]
    },
    {
        "type": "Correct Prediction 2",
        "context": "one of the",
        "expected": "most",
        "model_pred": predict_next_word("one of the", top_k=1)[0][0],
        "prob": predict_next_word("one of the", top_k=1)[0][1]
    },
    {
        "type": "Incorrect Prediction 1",
        "context": "the cat",
        "expected": "was",
        "model_pred": predict_next_word("the cat", top_k=1)[0][0],
        "prob": predict_next_word("the cat", top_k=1)[0][1]
    },
    {
        "type": "Incorrect Prediction 2",
        "context": "machine learning",
        "expected": "models",
        "model_pred": predict_next_word("machine learning", top_k=1)[0][0],
        "prob": predict_next_word("machine learning", top_k=1)[0][1]
    }
]

print(f"{'Type':<25}{'Context':<20}{'Model Pred':<15}{'Expected':<15}{'Prob':<10}")
print("-" * 85)
for c in cases:
    print(f"{c['type']:<25}{c['context']:<20}{c['model_pred']:<15}{c['expected']:<15}{c['prob']:<10.4f}")
print("-" * 85)


Type                     Context             Model Pred     Expected       Prob      
-------------------------------------------------------------------------------------
Correct Prediction 1     natural language    processing     processing     0.4000    
Correct Prediction 2     one of the          most           most           0.3700    
Incorrect Prediction 1   the cat             ,              was            0.2400    
Incorrect Prediction 2   machine learning    and            models         0.3500    
-------------------------------------------------------------------------------------


## 22.1. Phân tích nguyên nhân lỗi (Error Analysis)

### 1. Trường hợp đúng 1 (Correct Prediction 1)
- **Context:** `natural language`
- **Model prediction:** `processing`
- **Expected:** `processing`
- **Probability:** $0.4000$
- **Xác định nguyên nhân thành công:**
  - *Collocation & Insufficient training data (được thỏa mãn):* Cụm từ `natural language processing` là thuật ngữ chuẩn hóa xuất hiện với tần suất lặp lại rất cao trong tập huấn luyện.
  - *Sparsity thấp ở ngữ cảnh hẹp:* Số lượng từ đi sau cụm `natural language` rất ít và tập trung cao độ, phân phối xác suất sắc nhọn (sharp distribution), giúp mô hình dễ dàng vượt qua ngưỡng nhiễu.

### 2. Trường hợp đúng 2 (Correct Prediction 2)
- **Context:** `one of the`
- **Model prediction:** `most`
- **Expected:** `most`
- **Probability:** $0.3700$
- **Xác định nguyên nhân thành công:**
  - *Cấu trúc ngữ pháp phổ biến (Grammatical template):* `one of the most` là một cấu trúc cố định trong tiếng Anh. Tần suất xuất hiện trong tập train là hàng nghìn lần, áp đảo hoàn toàn các từ loại khác.
  - *Độ dài ngữ cảnh phù hợp:* Trigram context gồm 2 từ `one of` kết hợp với `the` đã đủ để hạn chế không gian tìm kiếm xuống các tính từ so sánh nhất.

### 3. Trường hợp sai 1 (Incorrect Prediction 1)
- **Context:** `the cat`
- **Model prediction:** `,`
- **Expected:** `was`
- **Probability:** $0.2400$
- **Xác định nguyên nhân thất bại:**
  - **Context quá ngắn:** Context chỉ gồm 2 từ `the cat` chỉ đóng vai trò một danh từ đơn lẻ. Trong tiếng Anh, sau một danh từ chủ ngữ có thể là mệnh đề quan hệ bắt đầu bằng dấu phẩy, sở hữu cách (`'s`), hoặc hàng nghìn động từ khác nhau. Ngữ cảnh quá ngắn không đủ để xác định hành động vị ngữ cụ thể.
  - **Preprocessing & Tokenization:** Việc tokenizer giữ nguyên các dấu câu (`,`, `.`) thành các token độc lập trong từ vựng, kết hợp với phong cách viết của kho ngữ liệu mạng (C4) thường xuyên có các mệnh đề bổ ngữ sau dấu phẩy, dẫn đến dấu phẩy chiếm tần suất thống kê cục bộ lớn hơn bất kỳ động từ cụ thể nào.
  - **Data Sparsity:** Các câu cụ thể về con mèo (`the cat was`, `the cat sat`) xuất hiện phân tán, không có mẫu câu nào chiếm ưu thế vượt trội để lấn át dấu câu.

### 4. Trường hợp sai 2 (Incorrect Prediction 2)
- **Context:** `machine learning`
- **Model prediction:** `and`
- **Expected:** `models`
- **Probability:** $0.3500$
- **Xác định nguyên nhân thất bại:**
  - **Dominance of Stop Words & High-frequency unigrams:** Từ nối `and` là một trong những từ phổ biến nhất trong toàn bộ ngữ liệu. Trong văn phong kỹ thuật, `machine learning and ...` (liệt kê cùng `artificial intelligence`, `data science`) xuất hiện nhiều hơn bất kỳ danh từ đơn lẻ đi kèm nào.
  - **Smoothing Effect:** Mô hình Laplace làm phẳng nhẹ phân phối, nhưng đối với các từ có count cao như `and`, sự vượt trội về tần suất bề mặt vẫn lấn át các từ chuyên ngành có tính ngữ nghĩa chính xác hơn như `models` hay `algorithms`.
  - **Vocabulary limitation / Domain mismatch:** Kho ngữ liệu C4 là tập văn bản web tổng hợp, không phải một corpus chuyên sâu về khoa học máy tính, nên các cụm chuyên ngành chưa đủ độ tập trung áp đảo so với văn phong thông thường.


# 23. Một thí nghiệm rất quan trọng: Context Length
Khảo sát định lượng tác động của độ dài ngữ cảnh (Context Length) từ Unigram đến Trigram và lý giải bước chuyển dịch mô hình trong NLP.


In [23]:
total_valid_unigrams = sum(len(s) - 1 for s in valid_corpus)
total_valid_bigrams = sum(len(s) - 1 for s in valid_corpus)
total_valid_trigrams = sum(max(0, len(s) - 2) for s in valid_corpus)

unseen_valid_unigrams = sum(1 for s in valid_corpus for i in range(1, len(s)) if s[i] == "<UNK>")
unseen_valid_bigrams = sum(1 for s in valid_corpus for i in range(1, len(s)) if bigram_counts[(s[i-1], s[i])] == 0)
unseen_valid_trigrams = sum(1 for s in valid_corpus for i in range(2, len(s)) if trigram_counts[(s[i-2], s[i-1], s[i])] == 0)

print(f"{'Order':<15}{'Train N-grams':>18}{'Total Valid Tokens':>22}{'Unseen in Valid':>18}{'Unseen Ratio (%)':>18}")
print("-" * 95)
print(f"{'Unigram (N=1)':<15}{len(unigram_counts):>18}{total_valid_unigrams:>22}{unseen_valid_unigrams:>18}{unseen_valid_unigrams / total_valid_unigrams * 100:>17.2f}%")
print(f"{'Bigram (N=2)':<15}{len(bigram_counts):>18}{total_valid_bigrams:>22}{unseen_valid_bigrams:>18}{unseen_valid_bigrams / total_valid_bigrams * 100:>17.2f}%")
print(f"{'Trigram (N=3)':<15}{len(trigram_counts):>18}{total_valid_trigrams:>22}{unseen_valid_trigrams:>18}{unseen_valid_trigrams / total_valid_trigrams * 100:>17.2f}%")
print("-" * 95)


Order                 Train N-grams    Total Valid Tokens   Unseen in Valid  Unseen Ratio (%)
-----------------------------------------------------------------------------------------------
Unigram (N=1)                307578               1246988              7289              0.58%
Bigram (N=2)                2242028               1246988            547432             43.90%
Trigram (N=3)               6762788               1184768           1018900             86.00%
-----------------------------------------------------------------------------------------------


## 23.1. Phân tích thực nghiệm: Context dài hơn có luôn tốt hơn không?

### 1. Trả lời câu hỏi
**KHÔNG, trong mô hình ngôn ngữ N-gram truyền thống, context dài hơn KHÔNG PHẢI LUÔN LUÔN TỐT HƠN.**

Dựa trên các bằng chứng định lượng từ thực nghiệm:
1. **Training Perplexity:**
   - Với MLE, Training Perplexity giảm đơn điệu theo độ dài context: Bigram ($118.59$) $	o$ Trigram ($9.93$). Khi context càng dài, mô hình càng dễ ghi nhớ dữ liệu huấn luyện.
2. **Validation & Test Perplexity:**
   - Hoàn toàn trái ngược với tập huấn luyện, trên Validation và Test:
     - Với MLE: PPL bằng $\infty$ (`inf`) ngay từ Bigram do zero-frequency.
     - Với Laplace: PPL bùng nổ theo cấp số nhân: Unigram ($1870.07$) $	o$ Bigram ($6938.85$) $	o$ Trigram ($82461.32$).
   - Như vậy, context dài hơn khiến chất lượng mô hình trên dữ liệu thực tế giảm sút thảm hại.
3. **Số lượng N-gram và Tỷ lệ Unseen N-gram:**
   - Bảng thống kê định lượng cho thấy rõ bản chất vấn đề:
     - **Unigram ($N=1$):** Tỷ lệ unseen token trên validation chỉ là **$0.58\%$**.
     - **Bigram ($N=2$):** Tỷ lệ unseen bigram tăng lên **$43.90\%$**.
     - **Trigram ($N=3$):** Tỷ lệ unseen trigram lên tới **$86.00\%$**!
   - Khi tăng bậc context, không gian trạng thái bùng nổ theo $V^N$, khiến phần lớn các tổ hợp gặp trong thực tế đều chưa từng xuất hiện trong tập huấn luyện (Data Sparsity).

---

## 23.2. Bước chuyển tự nhiên sang kỷ nguyên Mô hình Ngôn ngữ Hiện đại

Sự bế tắc của mô hình N-gram khi tăng context length đã thúc đẩy sự tiến hóa mang tính cách mạng của toàn bộ ngành Xử lý Ngôn ngữ Tự nhiên:

```text
       n-gram Language Models
                 ↓  (Bế tắc: Curse of Dimensionality & Data Sparsity)
     Neural Language Model (Bengio et al., 2003)
                 ↓  (Biểu diễn vector liên tục thay cho đếm rời rạc)
        Word Embeddings (Word2Vec, GloVe, 2013-2014)
                 ↓  (Không gian ngữ nghĩa dày đặc, từ tương đồng ở gần nhau)
           RNN / LSTM / GRU (2014-2016)
                 ↓  (Ngữ cảnh chuỗi có độ dài tùy ý, hidden state)
          Attention Mechanism (Bahdanau et al., 2015)
                 ↓  (Nhìn trực tiếp mọi vị trí trong quá khứ, giải phóng nút thắt cổ chai)
       Transformer Architecture (Vaswani et al., 2017) & Modern LLMs
```

### Chi tiết các mắt xích tiến hóa:

1. **Từ n-gram $	o$ Neural Language Model (Bengio et al., 2003):**
   - *Hạn chế của n-gram:* Biểu diễn rời rạc (discrete/one-hot). Hai câu `"The cat sits on the mat"` và `"The feline rests on the rug"` bị coi là hoàn toàn độc lập, không chia sẻ bất kỳ tham số nào.
   - *Giải pháp:* Bengio đề xuất ánh xạ mỗi từ thành một vector thực liên tục (continuous distributed representation) trong không gian nhiều chiều. Hàm xác suất được tính bằng mạng nơ-ron truyền thẳng (Feedforward Neural Network). Nhờ tính liên tục, các từ có ngữ nghĩa tương tự nhau sẽ có xác suất dự đoán tương đồng, giải quyết cơ bản vấn đề sparsity của n-gram.

2. **Từ Neural LM $	o$ Word Embeddings (Word2Vec, GloVe, 2013-2014):**
   - Huấn luyện không gian vector ngữ nghĩa dày đặc (Dense Semantic Vectors) trên kho dữ liệu khổng lồ với chi phí tính toán tối ưu (Skip-gram, CBOW).
   - Mô hình nắm bắt được quan hệ ngữ nghĩa tinh tế (ví dụ: $ec{v}_{	ext{King}} - ec{v}_{	ext{Man}} + ec{v}_{	ext{Woman}} pprox ec{v}_{	ext{Queen}}$), cho phép chia sẻ tri thức ngữ nghĩa trên toàn bộ từ vựng.

3. **Từ Word Embeddings $	o$ RNN / LSTM / GRU (2014-2016):**
   - *Hạn chế của mô hình cửa sổ cố định:* Cả n-gram và NNLM của Bengio đều bị giới hạn bởi một cửa sổ ngữ cảnh cố định $N$ từ (Markov assumption).
   - *Giải pháp:* Recurrent Neural Networks (RNN) và LSTM duy trì một trạng thái ẩn (hidden state $h_t$) được cập nhật liên tục qua từng bước thời gian:
     $$h_t = f(h_{t-1}, x_t)$$
     Về mặt lý thuyết, mạng nơ-ron hồi quy có thể ghi nhớ ngữ cảnh dài vô hạn mà số lượng tham số không hề tăng theo độ dài context.

4. **Từ RNN $	o$ Attention Mechanism (Bahdanau et al., 2015):**
   - *Hạn chế của RNN/LSTM:* Nén toàn bộ thông tin của chuỗi văn bản dài vào một vector ẩn duy nhất tạo ra "nút thắt cổ chai thông tin" (information bottleneck), đồng thời gặp hiện tượng triệt tiêu gradient (vanishing gradient) khi khoảng cách ngữ cảnh vượt quá 50-100 từ.
   - *Giải pháp:* Attention cho phép mô hình nhìn trực tiếp vào tất cả các trạng thái trong quá khứ và tính trọng số tương quan động:
     $$	ext{Attention}(Q, K, V) = 	ext{softmax}\left(rac{QK^T}{\sqrt{d_k}}
ight)V$$
     Mô hình có thể kết nối trực tiếp thông tin giữa hai từ cách xa nhau hàng trăm vị trí trong câu.

5. **Từ Attention $	o$ Transformer & Modern LLMs (Vaswani et al., 2017 $	o$ nay):**
   - Loại bỏ hoàn toàn kiến trúc tuần tự của RNN, dựa hoàn toàn vào **Self-Attention** kết hợp với Positional Encoding.
   - Cho phép tính toán song song hóa tuyệt đối trên GPU, giúp huấn luyện trên quy mô dữ liệu hàng nghìn tỷ token.
   - Mở rộng cửa sổ ngữ cảnh (Context Window) từ 2-3 từ của n-gram lên tới hàng trăm nghìn và hàng triệu token trong các mô hình nền tảng hiện đại (GPT-4, Claude, Gemini, LLaMA) với chất lượng dự đoán vượt trội và không bị vấp phải hiện tượng bùng nổ tổ hợp số chiều.
